# Midterm Project -- Group 2

## 一、数据导入与预览

In [1]:
import pandas as pd
import numpy as np
import re
import os
import joblib
from datetime import datetime
from sklearn.linear_model import SGDRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

In [2]:
# 数据导入
df_trainP = pd.read_csv('data/ruc_Class25Q2_train_price.csv', low_memory=False)
df_trainR = pd.read_csv('data/ruc_Class25Q2_train_rent.csv', low_memory=False)

col_map = {
    '户型': '房屋户型',
    '装修': '装修情况',
    '楼层': '所在楼层',
    '面积': '建筑面积',
    '朝向': '房屋朝向',
    '电梯': '配备电梯'
    }

df_trainR = df_trainR.rename(columns=col_map)

# 预览
print("Price数据结构:")
print(df_trainP.info())
print("\nRent数据结构:")
print(df_trainR.info())

Price数据结构:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 55 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   城市         103871 non-null  int64  
 1   区域         103871 non-null  float64
 2   板块         103871 non-null  float64
 3   环线         40419 non-null   object 
 4   Price      103871 non-null  float64
 5   房屋户型       103291 non-null  object 
 6   所在楼层       103871 non-null  object 
 7   建筑面积       103871 non-null  object 
 8   套内面积       35984 non-null   object 
 9   房屋朝向       103870 non-null  object 
 10  建筑结构       103291 non-null  object 
 11  装修情况       103291 non-null  object 
 12  梯户比例       101252 non-null  object 
 13  配备电梯       91520 non-null   object 
 14  别墅类型       1443 non-null    object 
 15  交易时间       103871 non-null  object 
 16  交易权属       103871 non-null  object 
 17  上次交易       78422 non-null   object 
 18  房屋用途       103870 non-null  object 
 19  房屋年限       5

## 二、数据准备

### 1. 划分数据集

In [3]:
# 为避免数据泄露，首先划分训练集与测试集
Xp = df_trainP.drop(['Price'], axis=1)
yp = df_trainP['Price']
Xr = df_trainR.drop(['Price'], axis=1)
yr = df_trainR['Price']

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, 
    yp, 
    test_size=0.2,
    random_state=111
)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, 
    yr, 
    test_size=0.2,
    random_state=111
)

print(f"Price训练集结构: Xp_train {Xp_train.shape}, yp_train {yp_train.shape}")
print(f"Price测试集结构: Xp_test {Xp_test.shape}, yp_test {yp_test.shape}")
print(f"\nRent训练集结构: Xr_train {Xr_train.shape}, yr_train {yr_train.shape}")
print(f"Rent测试集结构: Xr_test {Xr_test.shape}, yr_test {yr_test.shape}")

Price训练集结构: Xp_train (83096, 54), yp_train (83096,)
Price测试集结构: Xp_test (20775, 54), yp_test (20775,)

Rent训练集结构: Xr_train (79119, 45), yr_train (79119,)
Rent测试集结构: Xr_test (19780, 45), yr_test (19780,)


In [4]:
drop_col = ['年份', '客户反馈' ,'物业办公电话', '停车费用']

Xp_train = Xp_train.drop(columns=drop_col)
Xr_train = Xr_train.drop(columns=drop_col)
Xp_test = Xp_test.drop(columns=drop_col)
Xr_test = Xr_test.drop(columns=drop_col)

### 2. 数据预处理

In [5]:
# 创建预处理类（共同特征）
class Preprocessor:
    def __init__(self):
        pass
    
    def extract_room_info(self, house_o):
        """使用正则表达式提取房间数量"""
        house_o = str(house_o).replace(' ', '')
        
        room_match = re.search(r'(\d+)(?:室|房间)', house_o)
        living_match = re.search(r'(\d+)厅', house_o)
        kitchen_match = re.search(r'(\d+)厨', house_o)
        bathroom_match = re.search(r'(\d+)卫', house_o)
        
        rooms = int(room_match.group(1)) if room_match else 0
        living_rooms = int(living_match.group(1)) if living_match else 0
        kitchens = int(kitchen_match.group(1)) if kitchen_match else 0
        bathrooms = int(bathroom_match.group(1)) if bathroom_match else 0
        
        return rooms, living_rooms, kitchens, bathrooms
    
    def extract_time_info(self, time_o):
        """提取建筑年代"""
        time_o = str(time_o).replace(' ', '')
        time_match = re.search(r'(\d{4})年', time_o)
        time = int(time_match.group(1)) if time_match else 0
        return time
    
    def extract_fee_info(self, fee_o):
        """提取物业费（取平均值）"""
        fee_o = str(fee_o).replace(' ', '')

        fee_match = re.search(r'(\d+\.?\d*)-(\d+\.?\d*)元/月/㎡', fee_o)
        if fee_match:
            fee_lower = float(fee_match.group(1))
            fee_upper = float(fee_match.group(2))
            return (fee_lower + fee_upper) / 2
        
        single_match = re.search(r'(\d+\.?\d*)元/月/㎡', fee_o)
        if single_match:
            fee_val = float(single_match.group(1))
            return fee_val
        
        return np.nan
    
    def extract_fuel_info(self, fuel_o):
        """提取燃气费（取平均值）"""
        fuel_o = str(fuel_o).replace(' ', '')

        fuel_match = re.search(r'(\d+\.?\d*)-(\d+\.?\d*)元/m³', fuel_o)
        if fuel_match:
            fuel_lower = float(fuel_match.group(1))
            fuel_upper = float(fuel_match.group(2))
            return (fuel_lower + fuel_upper) / 2
        
        fsingle_match = re.search(r'(\d+\.?\d*)元/m³', fuel_o)
        if fsingle_match:
            fuel_val = float(fsingle_match.group(1))
            return fuel_val
        
        return np.nan
    
    def extract_heat_info(self, heat_o):
        """提取供热费（取平均值）"""
        heat_o = str(heat_o).replace(' ', '')

        heat_match = re.search(r'(\d+\.?\d*)-(\d+\.?\d*)元/㎡', heat_o)
        if heat_match:
            heat_lower = float(heat_match.group(1))
            heat_upper = float(heat_match.group(2))
            return (heat_lower + heat_upper) / 2
        
        hsingle_match = re.search(r'(\d+\.?\d*)元/㎡', heat_o)
        if hsingle_match:
            heat_val = float(hsingle_match.group(1))
            return heat_val
        
        return np.nan
    
    def preprocess(self, df, verbose=True):
        df_processed = df.copy()
        
        # 1. 房屋户型处理
        room_info = df_processed['房屋户型'].apply(self.extract_room_info)
        df_processed['室'] = room_info.apply(lambda x: x[0])
        df_processed['厅'] = room_info.apply(lambda x: x[1])
        df_processed['厨'] = room_info.apply(lambda x: x[2])
        df_processed['卫'] = room_info.apply(lambda x: x[3])
        df_processed = df_processed.drop(columns=['房屋户型'])
        
        # 2. 建筑面积处理
        df_processed['建筑面积'] = df_processed['建筑面积'].str.replace('㎡', '').astype(float)
        
        # 3. 配备电梯处理
        df_processed['配备电梯'] = df_processed['配备电梯'].map({'有': 1, '无': 0})
        
        # 4. 交易时间处理
        df_processed['交易时间'] = pd.to_datetime(df_processed['交易时间'])
        df_processed['交易年份'] = df_processed['交易时间'].dt.year
        df_processed['交易月份'] = df_processed['交易时间'].dt.month
        df_processed = df_processed.drop(columns=['交易时间'])
        
        # 5. 建筑年代处理
        df_processed['建成时间'] = df_processed['建筑年代'].apply(self.extract_time_info)
        df_processed = df_processed.drop(columns=['建筑年代'])
        
        # 6. 房屋总数处理
        df_processed['房屋总数'] = df_processed['房屋总数'].str.replace('户', '').astype(float)
        
        # 7. 楼栋总数处理
        df_processed['楼栋总数'] = df_processed['楼栋总数'].str.replace('栋', '').astype(float)
        
        # 8. 绿化率处理
        df_processed['绿 化 率'] = df_processed['绿 化 率'].str.replace('%', '').astype(float)
        df_processed['绿 化 率'] = df_processed['绿 化 率'] * 0.01
        
        # 9. 物业费处理
        df_processed['物业费平均'] = df_processed['物 业 费'].apply(self.extract_fee_info)
        df_processed = df_processed.drop(columns=['物 业 费'])
        
        # 10. 燃气费处理
        df_processed['燃气费平均'] = df_processed['燃气费'].apply(self.extract_fuel_info)
        df_processed = df_processed.drop(columns=['燃气费'])
        
        # 11. 供热费处理
        df_processed['供热费平均'] = df_processed['供热费'].apply(self.extract_heat_info)
        df_processed = df_processed.drop(columns=['供热费'])
        
        if verbose:
            print(f"原始特征数量: {len(df.columns)}")
            print(f"处理后特征数量: {len(df_processed.columns)}")
            print(f"删除的特征: {set(df.columns) - set(df_processed.columns)}")
            print(f"新增的特征: {set(df_processed.columns) - set(df.columns)}")
        
        return df_processed

In [6]:
# 执行预处理
if __name__ == "__main__":
    preprocessor = Preprocessor()
    
    Xp_train = preprocessor.preprocess(Xp_train)
    print("Price训练集数据预处理完成\n")
    Xp_test= preprocessor.preprocess(Xp_test)
    print("Price测试集数据预处理完成\n")
    Xr_train = preprocessor.preprocess(Xr_train)
    print("Rent训练集数据预处理完成\n")
    Xr_test= preprocessor.preprocess(Xr_test)
    print("Rent测试集数据预处理完成")

原始特征数量: 50
处理后特征数量: 54
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
Price训练集数据预处理完成

原始特征数量: 50
处理后特征数量: 54
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
Price测试集数据预处理完成

原始特征数量: 41
处理后特征数量: 45
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
Rent训练集数据预处理完成

原始特征数量: 41
处理后特征数量: 45
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
Rent测试集数据预处理完成


In [7]:
# 分别处理不同特征
def process_floor_p(df, verbose=True):
    def extract_floor_p(floor_o):
        """提取所在楼层"""
        floor_o = str(floor_o)
        
        # 地下室
        if '地下室' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('地下室', total_floor)
        
        # 底层
        elif '底层' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('底层', total_floor)
        
        # 顶层
        elif '顶层' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('顶层', total_floor)
        
        # 低/中/高楼层
        elif any(x in floor_o for x in ['低楼层', '中楼层', '高楼层']):
            floor_type = None
            if '低楼层' in floor_o: 
                floor_type = '低楼层'
            elif '中楼层' in floor_o: 
                floor_type = '中楼层'
            elif '高楼层' in floor_o: 
                floor_type = '高楼层'
            
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            
            return (floor_type, total_floor)

        return ('其他', None)
    
    df_processed = df.copy()
    
    # 提取楼层信息
    floor_info = df_processed['所在楼层'].apply(extract_floor_p)
    df_processed['楼层类型'] = floor_info.apply(lambda x: x[0])
    df_processed['总楼层'] = floor_info.apply(lambda x: x[1])
    df_processed = df_processed.drop(columns=['所在楼层'])
    
    if verbose:
        print(f"楼层类型分布:")
        print(df_processed['楼层类型'].value_counts())
        print(f"总楼层统计:")
        print(df_processed['总楼层'].describe())
    
    return df_processed

print("Price训练集楼层数据处理：")
Xp_train = process_floor_p(Xp_train)
print("\nPrice测试集楼层数据处理：")
Xp_test = process_floor_p(Xp_test)

Price训练集楼层数据处理：
楼层类型分布:
中楼层    29047
高楼层    25598
低楼层    24570
顶层      1797
底层      1520
地下室      564
Name: 楼层类型, dtype: int64
总楼层统计:
count    83096.00000
mean        18.25479
std         10.98071
min          0.00000
25%          7.00000
50%         18.00000
75%         28.00000
max         70.00000
Name: 总楼层, dtype: float64

Price测试集楼层数据处理：
楼层类型分布:
中楼层    7204
高楼层    6534
低楼层    6098
顶层      467
底层      334
地下室     138
Name: 楼层类型, dtype: int64
总楼层统计:
count    20775.000000
mean        18.341757
std         10.995631
min          0.000000
25%          7.000000
50%         18.000000
75%         28.000000
max         63.000000
Name: 总楼层, dtype: float64


In [8]:
def process_floor_r(df, verbose=True):
    def extract_floor_r(floor_o):
        """提取所在楼层"""
        floor_o = str(floor_o).strip()
        
        slash_match = re.search(r'(\d+)/(\d+)层', floor_o)
        if slash_match:
            current_floor = int(slash_match.group(1))
            total_floor = int(slash_match.group(2))
            
            # 根据当前楼层与总楼层的关系判断楼层类型
            if current_floor == 1:
                return ('底层', total_floor)
            elif current_floor == total_floor:
                return ('顶层', total_floor)
            else:
                floor_ratio = current_floor / total_floor
                if floor_ratio <= 0.33:
                    return ('低楼层', total_floor)
                elif floor_ratio <= 0.66:
                    return ('中楼层', total_floor)
                else:
                    return ('高楼层', total_floor)
        
        # 地下室
        elif '地下室' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('地下室', total_floor)
        
        # 底层
        elif '底层' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('底层', total_floor)
        
        # 顶层
        elif '顶层' in floor_o:
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            return ('顶层', total_floor)
        
        # 低/中/高楼层
        elif any(x in floor_o for x in ['低楼层', '中楼层', '高楼层']):
            floor_type = None
            if '低楼层' in floor_o: 
                floor_type = '低楼层'
            elif '中楼层' in floor_o: 
                floor_type = '中楼层'
            elif '高楼层' in floor_o: 
                floor_type = '高楼层'
            
            total_match = re.search(r'共(\d+)层', floor_o)
            total_floor = int(total_match.group(1)) if total_match else None
            
            return (floor_type, total_floor)

        return ('其他', None)
    
    df_processed = df.copy()
    
    # 提取楼层信息
    floor_info = df_processed['所在楼层'].apply(extract_floor_r)
    df_processed['楼层类型'] = floor_info.apply(lambda x: x[0])
    df_processed['总楼层'] = floor_info.apply(lambda x: x[1])
    df_processed = df_processed.drop(columns=['所在楼层'])
    
    if verbose:
        print(f"楼层类型分布:")
        print(df_processed['楼层类型'].value_counts())
        print(f"总楼层统计:")
        print(df_processed['总楼层'].describe())
    
    return df_processed

print("Rent训练集楼层数据处理：")
Xr_train = process_floor_r(Xr_train)
print("\nRent测试集楼层数据处理：")
Xr_test = process_floor_r(Xr_test)

Rent训练集楼层数据处理：
楼层类型分布:
中楼层    29820
高楼层    26407
低楼层    21807
顶层       622
底层       251
地下室      203
其他         9
Name: 楼层类型, dtype: int64
总楼层统计:
count    7343.000000
mean       19.279041
std        11.007286
min         1.000000
25%         8.000000
50%        18.000000
75%        29.000000
max        75.000000
Name: 总楼层, dtype: float64

Rent测试集楼层数据处理：
楼层类型分布:
中楼层    7389
高楼层    6594
低楼层    5500
顶层      183
底层       62
地下室      49
其他        3
Name: 楼层类型, dtype: int64
总楼层统计:
count    1843.000000
mean       19.285947
std        11.599036
min         1.000000
25%         7.000000
50%        18.000000
75%        30.000000
max        48.000000
Name: 总楼层, dtype: float64


In [9]:
# 处理目标变量（取对数，改善数据分布并改进模型性能）
yp_train= np.log(yp_train)
yr_train = np.log(yr_train)
yp_test = np.log(yp_test)
yr_test = np.log(yr_test)

### 3. 异常值处理

In [10]:
def detect_outliers_iqr_x_only(X0, y0, threshold=3):
    # 检测X异常值
    outlier_info_X = {}
    mask_X = pd.Series([True] * len(X0), index=X0.index)
    
    numeric_columns = X0.select_dtypes(include=[np.int64, np.float64]).columns
    
    for col in numeric_columns:
        Q1 = np.percentile(X0[col], 25)
        Q3 = np.percentile(X0[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        outliers_mask = (X0[col] < lower_bound) | (X0[col] > upper_bound)
        n_outliers = outliers_mask.sum()
        
        if n_outliers > 0:
            outlier_info_X[col] = {
                'n_outliers': n_outliers,
                'outlier_rate': n_outliers / len(X0) * 100,
                'bounds': [lower_bound, upper_bound],
                'actual_range': [X0[col].min(), X0[col].max()]
            }
            
            mask_X = mask_X & ~outliers_mask
    
    n_X_outliers = len(X0) - np.sum(mask_X)
    
    print(f"原始样本量: {len(X0)}")
    print(f"X数值列异常值样本数: {n_X_outliers} ({n_X_outliers/len(X0)*100:.2f}%)")
    
    if outlier_info_X:
        print("\nX异常值详情:")
        for col, info in outlier_info_X.items():
            print(f"  {col}:")
            print(f"    异常值数量: {info['n_outliers']} ({info['outlier_rate']:.2f}%)")
            print(f"    正常范围: [{info['bounds'][0]:.2f}, {info['bounds'][1]:.2f}]")
            print(f"    实际范围: [{info['actual_range'][0]:.2f}, {info['actual_range'][1]:.2f}]")
    else:
        print("\n未检测到异常值")
    
    return mask_X, outlier_info_X

print("Price数据集:")
mask_Xp, outlier_info_Xp = detect_outliers_iqr_x_only(Xp_train, yp_train)
print("\nRent数据集:")
mask_Xr, outlier_info_Xr = detect_outliers_iqr_x_only(Xr_train, yr_train)

Xp_train = Xp_train[mask_Xp]
yp_train = yp_train[mask_Xp]
Xr_train = Xr_train[mask_Xr]
yr_train = yr_train[mask_Xr]

Price数据集:
原始样本量: 83096
X数值列异常值样本数: 3772 (4.54%)

X异常值详情:
  建筑面积:
    异常值数量: 1160 (1.40%)
    正常范围: [-72.80, 262.71]
    实际范围: [11.70, 508.11]
  室:
    异常值数量: 154 (0.19%)
    正常范围: [-1.00, 6.00]
    实际范围: [0.00, 12.00]
  厅:
    异常值数量: 5 (0.01%)
    正常范围: [-2.00, 5.00]
    实际范围: [0.00, 9.00]
  厨:
    异常值数量: 2676 (3.22%)
    正常范围: [1.00, 1.00]
    实际范围: [0.00, 7.00]
  卫:
    异常值数量: 79 (0.10%)
    正常范围: [-2.00, 5.00]
    实际范围: [0.00, 12.00]
  交易年份:
    异常值数量: 19 (0.02%)
    正常范围: [2020.00, 2027.00]
    实际范围: [2018.00, 2025.00]

Rent数据集:
原始样本量: 79119
X数值列异常值样本数: 5137 (6.49%)

X异常值详情:
  建筑面积:
    异常值数量: 577 (0.73%)
    正常范围: [-98.26, 248.35]
    实际范围: [6.05, 440.00]
  室:
    异常值数量: 4 (0.01%)
    正常范围: [-5.00, 9.00]
    实际范围: [0.00, 12.00]
  厅:
    异常值数量: 1 (0.00%)
    正常范围: [-2.00, 5.00]
    实际范围: [0.00, 7.00]
  卫:
    异常值数量: 36 (0.05%)
    正常范围: [-3.00, 4.00]
    实际范围: [0.00, 9.00]
  交易年份:
    异常值数量: 4566 (5.77%)
    正常范围: [2024.00, 2024.00]
    实际范围: [2024.00, 2025.00]


In [11]:
def detect_outliers_iqr_y_only(y0, threshold=3):
    Q1_y = np.percentile(y0, 25)
    Q3_y = np.percentile(y0, 75)
    IQR_y = Q3_y - Q1_y
    lower_bound_y = Q1_y - threshold * IQR_y
    upper_bound_y = Q3_y + threshold * IQR_y
    
    mask_y = (y0 >= lower_bound_y) & (y0 <= upper_bound_y)
    n_outliers_y = len(y0) - np.sum(mask_y)
    
    outlier_info = {
        'n_outliers': n_outliers_y,
        'outlier_rate': n_outliers_y / len(y0) * 100,
        'bounds': [lower_bound_y, upper_bound_y],
        'actual_range': [y0.min(), y0.max()]
    }
    
    print(f"y原始样本量: {len(y0)}")
    print(f"y异常值数量: {n_outliers_y} ({n_outliers_y/len(y0)*100:.2f}%)")
    print(f"y正常范围: [{lower_bound_y:.2f}, {upper_bound_y:.2f}]")
    print(f"y实际范围: [{y0.min():.2f}, {y0.max():.2f}]")
    
    return mask_y, outlier_info

print("Price数据集:")
mask_yp, info_yp = detect_outliers_iqr_y_only(yp_train)
print("\nRent数据集:")
mask_yr, info_yr = detect_outliers_iqr_y_only(yr_train)

Xp_train = Xp_train[mask_yp]
yp_train = yp_train[mask_yp]
Xr_train = Xr_train[mask_yr]
yr_train = yr_train[mask_yr]

Price数据集:
y原始样本量: 79324
y异常值数量: 0 (0.00%)
y正常范围: [10.48, 18.00]
y实际范围: [11.60, 17.15]

Rent数据集:
y原始样本量: 73982
y异常值数量: 0 (0.00%)
y正常范围: [9.11, 16.77]
y实际范围: [9.79, 15.95]


### 4. 特征工程

In [12]:
# 添加非线性特征和交互项
def create_features(df):
    df_new = df.copy()

    df_new['建筑面积2'] = df_new['建筑面积'] ** 2
    df_new['总楼层2'] = df_new['总楼层'] ** 2
    df_new['面积_室交互'] = df_new['建筑面积'] * df_new['室']
    df_new['面积_厅交互'] = df_new['建筑面积'] * df_new['厅']
    df_new['面积_厨交互'] = df_new['建筑面积'] * df_new['厨']
    df_new['面积_卫交互'] = df_new['建筑面积'] * df_new['卫']
    df_new['楼层_电梯交互'] = df_new['总楼层'] * df_new['配备电梯']
    df_new['位置交互'] = df_new['coord_x'] * df_new['coord_y']
    
    return df_new

Xp_train = create_features(Xp_train)
Xp_test = create_features(Xp_test)
Xr_train = create_features(Xr_train)
Xr_test = create_features(Xr_test)

## 三、模型训练

### 1. 管道构建

In [13]:
# Price处理管道（缺失值填充、类别变量编码）
cat_features_p = Xp_train.select_dtypes(include='object').columns
num_features_p = Xp_train.select_dtypes(include=np.number).columns

cat_pipe_p = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=True))
])

num_pipe_p = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

preprocessor_p = ColumnTransformer([
    ('cat', cat_pipe_p, cat_features_p),
    ('num', num_pipe_p, num_features_p)
])

In [14]:
# Rent处理管道（缺失值填充、类别变量编码）
cat_features_r = Xr_train.select_dtypes(include='object').columns
num_features_r = Xr_train.select_dtypes(include=np.number).columns

cat_pipe_r = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=True))
])

num_pipe_r = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

preprocessor_r = ColumnTransformer([
    ('cat', cat_pipe_r, cat_features_r),
    ('num', num_pipe_r, num_features_r)
])

### 2. 模型构建与训练（Stochastic Gradient Descent）

In [15]:
# Price建模
# 1. OLS
ols_pipeline_p = Pipeline([
    ('preprocessor', preprocessor_p),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty=None,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,        # 启用早停策略以防止过拟合
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 2. Lasso
lasso_pipeline_p = Pipeline([
    ('preprocessor', preprocessor_p),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='l1',
        alpha=0.001,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 3. Ridge
ridge_pipeline_p = Pipeline([
    ('preprocessor', preprocessor_p),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='l2',
        alpha=0.001,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 4. Elastic Net
elastic_pipeline_p = Pipeline([
    ('preprocessor', preprocessor_p),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='elasticnet',
        alpha=0.001,
        l1_ratio=0.5,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# Rent建模
# 1. OLS
ols_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty=None,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 2. Lasso
lasso_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='l1',
        alpha=0.001,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 3. Ridge
ridge_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='l2',
        alpha=0.001,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# 4. Elastic Net
elastic_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('regressor', SGDRegressor(
        loss='squared_error',
        penalty='elasticnet',
        alpha=0.001,
        l1_ratio=0.5,
        learning_rate='invscaling',
        eta0=0.01,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# Price模型集
models_p = {
    'OLS_p': ols_pipeline_p,
    'Lasso_p': lasso_pipeline_p,
    'Ridge_p': ridge_pipeline_p,
    'ElasticNet_p': elastic_pipeline_p
}

# Rent模型集
models_r = {
    'OLS_r': ols_pipeline_r,
    'Lasso_r': lasso_pipeline_r,
    'Ridge_r': ridge_pipeline_r,
    'ElasticNet_r': elastic_pipeline_r
}

# 训练Price模型
for name, model in models_p.items():
    model.fit(Xp_train, yp_train)
    print(f"{name} 训练完成")

# 训练Rent模型
for name, model in models_r.items():
    model.fit(Xr_train, yr_train)
    print(f"{name} 训练完成")

# 模型评估（计算MAE、RMAE）
def evaluate_models_rmae(models, X_train, X_test, y_train, y_test, dataset_name=""):
    results = {}
    for name, model in models.items():
        y_train_pred = model.predict(X_train)
        train_mae = mean_absolute_error(y_train, y_train_pred)
        train_rmae = train_mae / np.mean(np.abs(y_train))
        
        y_test_pred = model.predict(X_test)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rmae = test_mae / np.mean(np.abs(y_test))
        
        regressor = model.named_steps['regressor']
        results[name] = {
            'Train_MAE': train_mae,
            'Train_RMAE': train_rmae,
            'Test_MAE': test_mae,
            'Test_RMAE': test_rmae,
            'Iterations': regressor.n_iter_,
            'Early_Stopped': regressor.n_iter_ < regressor.max_iter
        }
        
        prefix = f"[{dataset_name}] " if dataset_name else ""
        print(f"{prefix}{name}:")
        print(f"  训练集 MAE: {train_mae:.4f}")
        print(f"  训练集 RMAE: {train_rmae:.4f} ({train_rmae*100:.2f}%)")
        print(f"  测试集 MAE: {test_mae:.4f}")
        print(f"  测试集 RMAE: {test_rmae:.4f} ({test_rmae*100:.2f}%)")
        print(f"  迭代次数: {regressor.n_iter_}")
        print(f"  是否早停: {'是' if regressor.n_iter_ < regressor.max_iter else '否'}")
        print("-" * 60)
    
    return results

print("\nPrice模型训练结果:")
results_p = evaluate_models_rmae(models_p, Xp_train, Xp_test, yp_train, yp_test, "Price")

print("\nRent模型训练结果:")
results_r = evaluate_models_rmae(models_r, Xr_train, Xr_test, yr_train, yr_test, "Rent")

# 确定最佳模型
best_model_name_p = min(results_p.items(), key=lambda x: x[1]['Test_RMAE'])[0]
best_rmae_p = results_p[best_model_name_p]['Test_RMAE']

best_model_name_r = min(results_r.items(), key=lambda x: x[1]['Test_RMAE'])[0]
best_rmae_r = results_r[best_model_name_r]['Test_RMAE']

print(f"\nPrice最佳模型: {best_model_name_p}")
print(f"Price最佳测试集RMAE: {best_rmae_p:.4f} ({best_rmae_p*100:.2f}%)")

print(f"\nRent最佳模型: {best_model_name_r}")
print(f"Rent最佳测试集RMAE: {best_rmae_r:.4f} ({best_rmae_r*100:.2f}%)")

# 保存OLS模型
joblib.dump(ols_pipeline_p, os.path.join("ols_model_p.pkl"))
joblib.dump(ols_pipeline_r, os.path.join("ols_model_r.pkl"))

OLS_p 训练完成
Lasso_p 训练完成
Ridge_p 训练完成
ElasticNet_p 训练完成
OLS_r 训练完成
Lasso_r 训练完成
Ridge_r 训练完成
ElasticNet_r 训练完成

Price模型训练结果:
[Price] OLS_p:
  训练集 MAE: 0.1950
  训练集 RMAE: 0.0137 (1.37%)
  测试集 MAE: 0.2215
  测试集 RMAE: 0.0155 (1.55%)
  迭代次数: 41
  是否早停: 是
------------------------------------------------------------
[Price] Lasso_p:
  训练集 MAE: 0.2992
  训练集 RMAE: 0.0210 (2.10%)
  测试集 MAE: 0.3118
  测试集 RMAE: 0.0219 (2.19%)
  迭代次数: 32
  是否早停: 是
------------------------------------------------------------
[Price] Ridge_p:
  训练集 MAE: 0.2533
  训练集 RMAE: 0.0178 (1.78%)
  测试集 MAE: 0.2699
  测试集 RMAE: 0.0189 (1.89%)
  迭代次数: 11
  是否早停: 是
------------------------------------------------------------
[Price] ElasticNet_p:
  训练集 MAE: 0.2963
  训练集 RMAE: 0.0208 (2.08%)
  测试集 MAE: 0.3082
  测试集 RMAE: 0.0216 (2.16%)
  迭代次数: 11
  是否早停: 是
------------------------------------------------------------

Rent模型训练结果:
[Rent] OLS_r:
  训练集 MAE: 0.1789
  训练集 RMAE: 0.0138 (1.38%)
  测试集 MAE: 0.1861
  测试集 RMAE: 0.0144 (1.44%)


['ols_model_r.pkl']

### 3. 六重交叉验证

In [16]:
def cv_mae(models_p, models_r, Xp_train, yp_train, Xr_train, yr_train, cv_folds=6):
    # Price模型评估
    print("\nPrice模型交叉验证结果 (MAE):")
    print("=" * 50)
    for name, model in models_p.items():
        mae_scores = -cross_val_score(
            model, Xp_train, yp_train, 
            cv=cv_folds, 
            scoring='neg_mean_absolute_error'
        )
        
        mean_mae = np.mean(mae_scores)
        std_mae = np.std(mae_scores)
        
        print(f"{name:15} | MAE: {mean_mae:>8.2f} (±{std_mae:.2f})")
    
    # Rent模型评估
    print("\nRent模型交叉验证结果 (MAE):")
    print("=" * 50)
    for name, model in models_r.items():
        mae_scores = -cross_val_score(
            model, Xr_train, yr_train, 
            cv=cv_folds, 
            scoring='neg_mean_absolute_error'
        )
        
        mean_mae = np.mean(mae_scores)
        std_mae = np.std(mae_scores)
        
        print(f"{name:15} | MAE: {mean_mae:>8.2f} (±{std_mae:.2f})")

cv_mae(models_p, models_r, Xp_train, yp_train, Xr_train, yr_train)


Price模型交叉验证结果 (MAE):
OLS_p           | MAE:     0.22 (±0.02)
Lasso_p         | MAE:     0.30 (±0.01)
Ridge_p         | MAE:     0.26 (±0.00)
ElasticNet_p    | MAE:     0.30 (±0.00)

Rent模型交叉验证结果 (MAE):
OLS_r           | MAE:     0.19 (±0.01)
Lasso_r         | MAE:     0.27 (±0.00)
Ridge_r         | MAE:     0.23 (±0.00)
ElasticNet_r    | MAE:     0.27 (±0.00)


### 4. 模型调优

In [17]:
# 1.定义调优函数
def quick_tune_lasso(X_train, y_train, preprocessor, model_name="Lasso"):
    print(f"开始调优{model_name}模型...")
    
    lasso_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', SGDRegressor(
            loss='squared_error',
            penalty='l1',
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
    
    param_grid = {
        'regressor__alpha': [0.001, 0.01, 0.1],
        'regressor__eta0': [0.01, 0.1],
        'regressor__learning_rate': ['invscaling', 'constant']
    }
    
    grid_search = GridSearchCV(
        lasso_pipeline,
        param_grid,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"{model_name}最佳参数: {grid_search.best_params_}")
    print(f"{model_name}最佳MAE: {-grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

def quick_tune_ridge(X_train, y_train, preprocessor, model_name="Ridge"):
    print(f"开始调优{model_name}模型...")
    
    ridge_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', SGDRegressor(
            loss='squared_error',
            penalty='l2',
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
    
    param_grid = {
        'regressor__alpha': [0.001, 0.01, 0.1],
        'regressor__eta0': [0.01, 0.1],
        'regressor__learning_rate': ['invscaling', 'constant']
    }
    
    grid_search = GridSearchCV(
        ridge_pipeline,
        param_grid,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"{model_name}最佳参数: {grid_search.best_params_}")
    print(f"{model_name}最佳MAE: {-grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

def quick_tune_en(X_train, y_train, preprocessor, model_name="ElasticNet"):
    print(f"开始调优{model_name}模型...")
    
    en_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', SGDRegressor(
            loss='squared_error',
            penalty='elasticnet',
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
    
    param_grid = {
        'regressor__alpha': [0.001, 0.01, 0.1],
        'regressor__l1_ratio': [0.3, 0.5, 0.7],
        'regressor__eta0': [0.01, 0.1],
        'regressor__learning_rate': ['invscaling', 'constant']
    }
    
    grid_search = GridSearchCV(
        en_pipeline,
        param_grid,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"{model_name}最佳参数: {grid_search.best_params_}")
    print(f"{model_name}最佳MAE: {-grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# 2. 评估函数（MAE）
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"{name} - 测试集MAE: {mae:.4f}")
    return mae

# 3. 执行调优
print("开始模型调优过程...")
tuned_models = {}

# Price模型调优
print("\nPRICE模型调优")
tuned_models['lasso_price'] = quick_tune_lasso(Xp_train, yp_train, preprocessor_p, "Price Lasso")
tuned_models['ridge_price'] = quick_tune_ridge(Xp_train, yp_train, preprocessor_p, "Price Ridge")
tuned_models['en_price'] = quick_tune_en(Xp_train, yp_train, preprocessor_p, "Price ElasticNet")

# Rent模型调优
print("\nRENT模型调优")
tuned_models['lasso_rent'] = quick_tune_lasso(Xr_train, yr_train, preprocessor_r, "Rent Lasso")
tuned_models['ridge_rent'] = quick_tune_ridge(Xr_train, yr_train, preprocessor_r, "Rent Ridge")
tuned_models['en_rent'] = quick_tune_en(Xr_train, yr_train, preprocessor_r, "Rent ElasticNet")

# 4. 评估所有模型
print("\n模型评估结果")

print("\nPrice模型测试结果:")
price_results = {}
for name in ['lasso_price', 'ridge_price', 'en_price']:
    mae = evaluate_model(tuned_models[name], Xp_test, yp_test, name)
    price_results[name] = {'mae': mae, 'model': tuned_models[name]}

print("\nRent模型测试结果:")
rent_results = {}
for name in ['lasso_rent', 'ridge_rent', 'en_rent']:
    mae = evaluate_model(tuned_models[name], Xr_test, yr_test, name)
    rent_results[name] = {'mae': mae, 'model': tuned_models[name]}

# 5. 最佳模型选择（基于MAE）
def select_best_model(results, task_name):
    best_name = min(results.items(), key=lambda x: x[1]['mae'])[0]
    best_mae = results[best_name]['mae']
    best_model = results[best_name]['model']
    
    print(f"{task_name}最佳模型: {best_name}")
    print(f"最佳MAE: {best_mae:.4f}")
    
    return best_name, best_model, best_mae

print("\n最佳模型选择")
best_price_name, best_price_model, best_price_mae = select_best_model(price_results, "Price")
best_rent_name, best_rent_model, best_rent_mae = select_best_model(rent_results, "Rent")

# 6. 保存模型
model_dir = "tuned_models"
os.makedirs(model_dir, exist_ok=True)

for name, model in tuned_models.items():
    filename = os.path.join(model_dir, f"{name}_model.pkl")
    joblib.dump(model, filename)
    print(f"已保存: {filename}")

joblib.dump(best_price_model, os.path.join(model_dir, "best_price_model.pkl"))
joblib.dump(best_rent_model, os.path.join(model_dir, "best_rent_model.pkl"))

print(f"\n最佳Price模型已保存: best_price_model.pkl")
print(f"最佳Rent模型已保存: best_rent_model.pkl")

# 7. 保存模型信息（MAE）
model_info = {
    'best_price': {
        'name': best_price_name,
        'mae': best_price_mae,
        'parameters': best_price_model.named_steps['regressor'].get_params()
    },
    'best_rent': {
        'name': best_rent_name,
        'mae': best_rent_mae,
        'parameters': best_rent_model.named_steps['regressor'].get_params()
    }
}

joblib.dump(model_info, os.path.join(model_dir, "model_info.pkl"))
print("模型信息已保存: model_info.pkl")

print("\n模型调优和保存完成！")

开始模型调优过程...

PRICE模型调优
开始调优Price Lasso模型...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Price Lasso最佳参数: {'regressor__alpha': 0.001, 'regressor__eta0': 0.1, 'regressor__learning_rate': 'invscaling'}
Price Lasso最佳MAE: 0.2996
开始调优Price Ridge模型...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Price Ridge最佳参数: {'regressor__alpha': 0.001, 'regressor__eta0': 0.1, 'regressor__learning_rate': 'invscaling'}
Price Ridge最佳MAE: 0.2532
开始调优Price ElasticNet模型...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Price ElasticNet最佳参数: {'regressor__alpha': 0.001, 'regressor__eta0': 0.1, 'regressor__l1_ratio': 0.3, 'regressor__learning_rate': 'invscaling'}
Price ElasticNet最佳MAE: 0.2890

RENT模型调优
开始调优Rent Lasso模型...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Rent Lasso最佳参数: {'regressor__alpha': 0.001, 'regressor__eta0': 0.1, 'regressor__learning_rate': 'invscaling'}
Rent Lasso最佳MAE: 0.2700
开始调优Rent Ridge模型...
Fitting 3 folds for each of 12 can

In [21]:
# Metrics Table
metrics_p = {
    'Metrics': ['OLS', 'Lasso', 'Ridge', 'ElasticNet'],
    'In sample': [results_p['OLS_p']['Train_MAE'], results_p['Lasso_p']['Train_MAE'], results_p['Ridge_p']['Train_MAE'], results_p['ElasticNet_p']['Train_MAE']],
    'Out of sample': [results_p['OLS_p']['Test_MAE'], results_p['Lasso_p']['Test_MAE'], results_p['Ridge_p']['Test_MAE'], results_p['ElasticNet_p']['Test_MAE']],
    'Cross-validation': [0.22, 0.30, 0.26, 0.30],
    'Kaggle Score': [62.1, 52.9, 55.1, 50.4]
}

metrics_r = {
    'Metrics': ['OLS', 'Lasso', 'Ridge', 'ElasticNet'],
    'In sample': [results_r['OLS_r']['Train_MAE'], results_r['Lasso_r']['Train_MAE'], results_r['Ridge_r']['Train_MAE'], results_r['ElasticNet_r']['Train_MAE']],
    'Out of sample': [results_r['OLS_r']['Test_MAE'], results_r['Lasso_r']['Test_MAE'], results_r['Ridge_r']['Test_MAE'], results_r['ElasticNet_r']['Test_MAE']],
    'Cross-validation': [0.19, 0.27, 0.23, 0.27],
    'Kaggle Score': [62.1, 52.9, 55.1, 50.4]
}

df_metrics_p = pd.DataFrame(metrics_p)
df_metrics_r = pd.DataFrame(metrics_r)
print("Price:")
print(df_metrics_p)
print("\nRent:")
print(df_metrics_r)

Price:
      Metrics  In sample  Out of sample  Cross-validation  Kaggle Score
0         OLS   0.195048       0.221460              0.22          62.1
1       Lasso   0.299192       0.311822              0.30          52.9
2       Ridge   0.253297       0.269940              0.26          55.1
3  ElasticNet   0.296295       0.308190              0.30          50.4

Rent:
      Metrics  In sample  Out of sample  Cross-validation  Kaggle Score
0         OLS   0.178869       0.186117              0.19          62.1
1       Lasso   0.268156       0.269754              0.27          52.9
2       Ridge   0.229266       0.235477              0.23          55.1
3  ElasticNet   0.262566       0.266488              0.27          50.4


## 四、预测结果

In [19]:
# 数据准备
df_testP = pd.read_csv('data/ruc_Class25Q2_test_price.csv', low_memory=False)
df_testR = pd.read_csv('data/ruc_Class25Q2_test_rent.csv', low_memory=False)

df_testR = df_testR.rename(columns=col_map)

df_testP = df_testP.drop(columns=drop_col)
df_testR = df_testR.drop(columns=drop_col)

if __name__ == "__main__":
    preprocessor = Preprocessor()
    df_testP = preprocessor.preprocess(df_testP)
    df_testR = preprocessor.preprocess(df_testR)
    
df_testP = process_floor_p(df_testP)
df_testR = process_floor_r(df_testR)
df_testP = create_features(df_testP)
df_testR = create_features(df_testR)

原始特征数量: 51
处理后特征数量: 55
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
原始特征数量: 42
处理后特征数量: 46
删除的特征: {'交易时间', '供热费', '房屋户型', '建筑年代', '物 业 费', '燃气费'}
新增的特征: {'卫', '厅', '建成时间', '燃气费平均', '厨', '物业费平均', '交易月份', '供热费平均', '室', '交易年份'}
楼层类型分布:
中楼层    13214
高楼层    10130
低楼层     9302
顶层       720
底层       623
地下室       28
Name: 楼层类型, dtype: int64
总楼层统计:
count    34017.000000
mean        17.474322
std         10.797705
min          0.000000
25%          6.000000
50%         17.000000
75%         27.000000
max         58.000000
Name: 总楼层, dtype: float64
楼层类型分布:
中楼层    3614
高楼层    3230
低楼层    2666
顶层      154
底层       69
地下室      40
Name: 楼层类型, dtype: int64
总楼层统计:
count    1444.000000
mean       17.660665
std        10.509070
min         2.000000
25%         6.000000
50%        18.000000
75%        27.000000
max        48.000000
Name: 总楼层, dtype: float64


In [20]:
def predict_with_models():
    trained_ols_model_p = joblib.load('ols_model_p.pkl')
    trained_ols_model_r = joblib.load('ols_model_r.pkl')
    
    tuned_lasso_p = joblib.load('tuned_models/lasso_price_model.pkl')
    tuned_ridge_p = joblib.load('tuned_models/ridge_price_model.pkl')
    tuned_en_p = joblib.load('tuned_models/en_price_model.pkl')
    
    tuned_lasso_r = joblib.load('tuned_models/lasso_rent_model.pkl')
    tuned_ridge_r = joblib.load('tuned_models/ridge_rent_model.pkl')
    tuned_en_r = joblib.load('tuned_models/en_rent_model.pkl')

    price_models = {
        'OLS': trained_ols_model_p,
        'Lasso': tuned_lasso_p,
        'Ridge': tuned_ridge_p,
        'ElasticNet': tuned_en_p
    }

    rent_models = {
        'OLS': trained_ols_model_r,
        'Lasso': tuned_lasso_r,
        'Ridge': tuned_ridge_r,
        'ElasticNet': tuned_en_r
    }

    def predict_models(models_dict, X_data, data_name):
        print(f"\n{data_name}模型预测:")
        predictions = {}
        for model_name, model in models_dict.items():
            y_pred = model.predict(X_data)
            y_pred_exp = np.exp(y_pred)  # 指数转换
            predictions[model_name] = y_pred_exp
            print(f"  {model_name}: {y_pred_exp.min():.2f} - {y_pred_exp.max():.2f}")
        return predictions

    price_predictions = predict_models(price_models, df_testP, "Price")
    rent_predictions = predict_models(rent_models, df_testR, "Rent")

    def create_merged_results_for_model(model_name):
        # Price结果
        if 'ID' in df_testP.columns:
            price_df = pd.DataFrame({'ID': df_testP['ID'], 'Price': price_predictions[model_name]})
        else:
            price_df = pd.DataFrame({'ID': df_testP.index, 'Price': price_predictions[model_name]})
        
        # Rent结果
        if 'ID' in df_testR.columns:
            rent_df = pd.DataFrame({'ID': df_testR['ID'], 'Price': rent_predictions[model_name]})
        else:
            rent_df = pd.DataFrame({'ID': df_testR.index, 'Price': rent_predictions[model_name]})
        
        # 合并
        merged_df = pd.concat([price_df, rent_df], ignore_index=True)
        
        filename = f'{model_name}_merged_predictions.csv'
        merged_df.to_csv(filename, index=False)
        
        print(f"  {model_name:12} | 文件: {filename} | 记录数: {len(merged_df)}")
        
        return merged_df

    print("\n生成四个模型的合并结果文件:")
    print("-" * 50)
    
    all_merged_results = {}
    for model_name in price_models.keys():
        all_merged_results[model_name] = create_merged_results_for_model(model_name)

    print(f"\n四个文件已保存完成!")
    print(f"生成的文件:")
    for model_name in price_models.keys():
        print(f"   - {model_name}_merged_predictions.csv")

    return all_merged_results

# 执行预测
all_results = predict_with_models()

print(f"\n各模型结果统计:")
for model_name, df in all_results.items():
    print(f"{model_name}: {len(df)} 条记录, 价格范围: {df['Price'].min():.2f} - {df['Price'].max():.2f}")


Price模型预测:
  OLS: 41417.00 - 144665277.45
  Lasso: 107730.31 - 155853758.17
  Ridge: 48422.20 - 144939901.79
  ElasticNet: 41512.86 - 159748380.52

Rent模型预测:
  OLS: 45788.86 - 16480764.94
  Lasso: 61492.53 - 11489117.69
  Ridge: 58740.74 - 17848612.48
  ElasticNet: 63165.00 - 18326373.08

生成四个模型的合并结果文件:
--------------------------------------------------
  OLS          | 文件: OLS_merged_predictions.csv | 记录数: 43790
  Lasso        | 文件: Lasso_merged_predictions.csv | 记录数: 43790
  Ridge        | 文件: Ridge_merged_predictions.csv | 记录数: 43790
  ElasticNet   | 文件: ElasticNet_merged_predictions.csv | 记录数: 43790

四个文件已保存完成!
生成的文件:
   - OLS_merged_predictions.csv
   - Lasso_merged_predictions.csv
   - Ridge_merged_predictions.csv
   - ElasticNet_merged_predictions.csv

各模型结果统计:
OLS: 43790 条记录, 价格范围: 41417.00 - 144665277.45
Lasso: 43790 条记录, 价格范围: 61492.53 - 155853758.17
Ridge: 43790 条记录, 价格范围: 48422.20 - 144939901.79
ElasticNet: 43790 条记录, 价格范围: 41512.86 - 159748380.52
